# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/meer140/flyrank-ml-internship-project/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

**Lane**: Lane 2 — Refresh / Content Opportunity Scoring  
**Deliverable**: Hand-written transparent rule baseline, signal audits, ranked baseline queue exported to `work/outputs/baseline_action_score.csv`, top-20 skeptic review, and leakage verification.

## 1. My rule and its reason codes

### Plain-Words Rule Definition
A page is prioritized for content refresh review if it possesses **high search visibility** (`impressions_90d`) combined with either **freshness decay** (un-updated for over 90–180 days) or **search engagement inefficiency** (Page 1 ranking with CTR below 0.5%).

---

### Signal Verification Audits (Signal Checks)

#### Signal 1: Content Staleness (`days_since_last_update`) — Flag-Linked
- **Hypothesis**: Older un-updated content suffers from search decay and higher decline rates.
- **Verdict**: **CONFIRMED**
- **Findings**: Pages un-updated for 90–180 days exhibit a **61.1% decline rate** compared to **51.1%** for fresh content (<30 days). Staleness is a reliable risk vector.

#### Signal 2: Page 1 CTR Efficiency (`ctr` for `avg_position` $\le 10$) — Flag-Linked
- **Hypothesis**: High-ranking (Page 1) pages with unexpectedly low CTR suffer from meta/snippet mismatches and decline over time.
- **Verdict**: **CONFIRMED**
- **Findings**: Page 1 pages with CTR <0.2% suffer a **58.9% decline rate**, whereas high CTR (>5.0%) pages have only a **31.3% decline rate**.

#### Signal 3: Content Word Count / Depth (`word_count`) — Secondary Signal
- **Hypothesis**: "Thin content" (<300 words) decays faster than deep long-form content (>2500 words).
- **Verdict**: **OPPOSITE** / **MIXED**
- **Findings**: Deep long-form content (>2500 words) actually exhibits a **59.1% decline rate**, whereas medium-length pages (600–1200 words) maintain the lowest decline rate at **25.0%**. Simple "thin = decaying" assumptions fail; long-form guides age out when facts become stale.

In [1]:
import pandas as pd
import numpy as np

# Load starter dataset
df = pd.read_csv('../../data/raw/content_refresh_anonymized.csv')

# Ground truth label: is_declining_label (trend_direction == 'down')
df['is_declining_label'] = (df['trend_direction'].str.lower() == 'down').astype(int)

print(f"Total dataset size: {len(df):,} rows")
print(f"Overall Base Declining Rate: {df['is_declining_label'].mean():.4f} ({df['is_declining_label'].mean()*100:.2f}%)\n")

# --- Signal 1 Bucket Table: Staleness (days_since_last_update) ---
df['staleness_bucket'] = pd.cut(
    df['days_since_last_update'],
    bins=[-1, 30, 90, 180, 365, 99999],
    labels=['<30d (Fresh)', '30-90d', '90-180d', '180-365d', '>365d (Stale)']
)

s1_table = df.groupby('staleness_bucket', observed=False).agg(
    n=('is_declining_label', 'count'),
    declining_rate=('is_declining_label', 'mean'),
    avg_impressions=('impressions_90d', 'mean')
).reset_index()

print("==========================================================================")
print("SIGNAL 1 AUDIT: Content Staleness (days_since_last_update) -> Verdict: CONFIRMED")
print("==========================================================================")
print(s1_table.to_string(index=False))

# --- Signal 2 Bucket Table: Page 1 CTR Efficiency (ctr for avg_position <= 10) ---
page1 = df[(df['avg_position'] > 0) & (df['avg_position'] <= 10)].copy()
page1['ctr_bucket'] = pd.cut(
    page1['ctr'],
    bins=[-0.01, 0.2, 0.5, 1.0, 5.0, 100.0],
    labels=['<0.2% (Severe Low)', '0.2-0.5% (Low)', '0.5-1.0% (Medium)', '1.0-5.0% (Good)', '>5.0% (High)']
)

s2_table = page1.groupby('ctr_bucket', observed=False).agg(
    n=('is_declining_label', 'count'),
    declining_rate=('is_declining_label', 'mean'),
    avg_impressions=('impressions_90d', 'mean')
).reset_index()

print("\n==========================================================================")
print("SIGNAL 2 AUDIT: Page 1 CTR Efficiency (avg_position <= 10) -> Verdict: CONFIRMED")
print("==========================================================================")
print(s2_table.to_string(index=False))

# --- Signal 3 Bucket Table: Word Count Depth (word_count) ---
df['word_count_bucket'] = pd.cut(
    df['word_count'].fillna(0),
    bins=[-1, 300, 600, 1200, 2500, 999999],
    labels=['<300 words (Thin)', '300-600', '600-1200', '1200-2500', '>2500 words (Deep)']
)

s3_table = df.groupby('word_count_bucket', observed=False).agg(
    n=('is_declining_label', 'count'),
    declining_rate=('is_declining_label', 'mean'),
    avg_impressions=('impressions_90d', 'mean')
).reset_index()

print("\n==========================================================================")
print("SIGNAL 3 AUDIT: Word Count Depth (word_count) -> Verdict: OPPOSITE / MIXED")
print("==========================================================================")
print(s3_table.to_string(index=False))

Total dataset size: 30,000 rows
Overall Base Declining Rate: 0.5421 (54.21%)

SIGNAL 1 AUDIT: Content Staleness (days_since_last_update) -> Verdict: CONFIRMED
staleness_bucket     n  declining_rate  avg_impressions
    <30d (Fresh) 20480        0.511377      4199.614062
          30-90d   175        0.588571      6506.748571
         90-180d  9171        0.611057      7486.665140
        180-365d   169        0.467456      1206.893491
   >365d (Stale)     5        0.600000         8.200000

SIGNAL 2 AUDIT: Page 1 CTR Efficiency (avg_position <= 10) -> Verdict: CONFIRMED
        ctr_bucket    n  declining_rate  avg_impressions
<0.2% (Severe Low) 7453        0.588756      4973.664699
    0.2-0.5% (Low) 2938        0.578625     12889.342069
 0.5-1.0% (Medium) 1448        0.500691     12160.330801
   1.0-5.0% (Good)  786        0.491094      5554.688295
      >5.0% (High)  358        0.312849        28.851955

SIGNAL 3 AUDIT: Word Count Depth (word_count) -> Verdict: OPPOSITE / MIXED
 word

## 2. Build the ranked queue (writes the CSV)

### Scoring Methodology & Transparent Rule Formula

We encode a readable baseline score using normalized and percentile-ranked features:

$$\text{baseline\_refresh\_score} = 0.40 \cdot \text{visibility\_score} + 0.30 \cdot \text{freshness\_risk\_score} + 0.25 \cdot \text{position\_opp\_score} + 0.05 \cdot \text{depth\_gap\_score}$$

Where:
- $\text{visibility\_score} = \text{percentile\_rank}(\log(1 + \text{impressions\_90d}))$
- $\text{freshness\_risk\_score} = \text{percentile\_rank}(\text{days\_since\_last\_update})$
- $\text{position\_opp\_score} = (1 - \text{normalize}(\text{avg\_position})) \times \text{visibility\_score}$
- $\text{depth\_gap\_score} = (1 - \text{percentile\_rank}(\text{word\_count})) \times \text{visibility\_score}$

### Reason Code & Action Label Mapping

Each content item receives **ONE primary reason code** and **ONE suggested action label**:

| Primary Reason Code | Criterion | Action Label |
|---|---|---|
| `thin_visible_page` | `word_count < 600` AND `impressions_90d >= 250` | `expand_and_refresh` |
| `low_ctr_visible_page` | `avg_position <= 10` AND `ctr < 0.5%` AND `impressions_90d >= 500` | `refresh_and_review_ctr` |
| `stale_visible_page` | `days_since_last_update >= 180` AND `impressions_90d >= 500` | `refresh` |
| `page_one_decay_risk` | `avg_position <= 10` AND `content_age_days >= 180` | `refresh` |
| `high_traffic_decay_risk` | `impressions_90d >= 500` | `refresh` |
| `general_refresh_review` | Default fallback | `monitor` |

In [2]:
import os
import json

# Percentile rank helper
def percentile_rank(s):
    return s.rank(pct=True)

# Min-max normalization helper
def normalize(s):
    s_min, s_max = s.min(), s.max()
    return (s - s_min) / (s_max - s_min) if s_max > s_min else s * 0

# Compute feature components
df['visibility_score'] = percentile_rank(np.log1p(df['impressions_90d']))
df['freshness_risk_score'] = percentile_rank(df['days_since_last_update'])
df['position_opp_score'] = (
    (1 - normalize(df['avg_position'].clip(lower=1, upper=50)))
    * df['visibility_score']
    * (df['avg_position'] > 0).astype(int)
)
df['depth_gap_score'] = (1 - percentile_rank(df['word_count'].fillna(0))) * df['visibility_score']

# Compute transparent score
df['baseline_refresh_score'] = (
    0.40 * df['visibility_score']
    + 0.30 * df['freshness_risk_score']
    + 0.25 * df['position_opp_score']
    + 0.05 * df['depth_gap_score']
).clip(0, 1)

# Primary Reason Code Logic
def assign_primary_reason(row):
    if row['days_since_last_update'] >= 180 and row['impressions_90d'] >= 500:
        return "stale_visible_page"
    if row['word_count'] > 0 and row['word_count'] < 600 and row['impressions_90d'] >= 250:
        return "thin_visible_page"
    if 0 < row['avg_position'] <= 10 and row['ctr'] < 0.5 and row['impressions_90d'] >= 500:
        return "low_ctr_visible_page"
    if 0 < row['avg_position'] <= 10 and row['content_age_days'] >= 180:
        return "page_one_decay_risk"
    if row['impressions_90d'] >= 500:
        return "high_traffic_decay_risk"
    return "general_refresh_review"

ACTION_MAP = {
    "thin_visible_page": "expand_and_refresh",
    "low_ctr_visible_page": "refresh_and_review_ctr",
    "stale_visible_page": "refresh",
    "page_one_decay_risk": "refresh",
    "high_traffic_decay_risk": "refresh",
    "general_refresh_review": "monitor"
}

df['reason_code'] = df.apply(assign_primary_reason, axis=1)
df['action_label'] = df['reason_code'].map(ACTION_MAP)

# Rank everything
df['baseline_rank'] = df['baseline_refresh_score'].rank(method='first', ascending=False).astype(int)

# Sort queue by rank
ranked_queue = df.sort_values('baseline_rank').copy()

# Output columns required for export
output_cols = [
    'content_id', 'client_id', 'baseline_rank', 'baseline_refresh_score',
    'visibility_score', 'freshness_risk_score', 'position_opp_score', 'depth_gap_score',
    'reason_code', 'action_label', 'is_declining_label',
    'impressions_90d', 'clicks_90d', 'sessions_90d', 'avg_position', 'ctr',
    'engagement_rate', 'scroll_rate', 'content_age_days', 'days_since_last_update', 'word_count'
]

# Ensure output directory exists
os.makedirs('../outputs', exist_ok=True)
csv_output_path = '../outputs/baseline_action_score.csv'
ranked_queue[output_cols].to_csv(csv_output_path, index=False)
print(f"Wrote baseline action queue to: {csv_output_path} ({len(ranked_queue):,} rows)")

# Calculate Precision@K vs Base Rate
base_rate = float(ranked_queue['is_declining_label'].mean())
p_at_10 = float(ranked_queue.head(10)['is_declining_label'].mean())
p_at_20 = float(ranked_queue.head(20)['is_declining_label'].mean())
p_at_50 = float(ranked_queue.head(50)['is_declining_label'].mean())

print(f"Base Rate (Full Dataset Declining Rate): {base_rate:.4f} ({base_rate*100:.1f}%)")
print(f"Baseline Precision@10: {p_at_10:.4f}")
print(f"Baseline Precision@20: {p_at_20:.4f}")
print(f"Baseline Precision@50: {p_at_50:.4f}")

# Save metrics metadata JSON receipt
json_metadata_path = '../outputs/baseline_metadata.json'
metadata = {
    "rows": len(ranked_queue),
    "base_declining_rate": base_rate,
    "top_score": float(ranked_queue['baseline_refresh_score'].max()),
    "median_score": float(ranked_queue['baseline_refresh_score'].median()),
    "precision_at_10": p_at_10,
    "precision_at_20": p_at_20,
    "precision_at_50": p_at_50,
    "score_weights": {
        "visibility_score": 0.40,
        "freshness_risk_score": 0.30,
        "position_opp_score": 0.25,
        "depth_gap_score": 0.05
    }
}
with open(json_metadata_path, 'w') as f:
    json.dump(metadata, f, indent=2)
print(f"Wrote metadata JSON to: {json_metadata_path}")

Wrote baseline action queue to: ../outputs/baseline_action_score.csv (30,000 rows)
Base Rate (Full Dataset Declining Rate): 0.5421 (54.2%)
Baseline Precision@10: 0.2000
Baseline Precision@20: 0.3500
Baseline Precision@50: 0.3400
Wrote metadata JSON to: ../outputs/baseline_metadata.json


## 3. Top-20 review

### Hand-Review of Top Picks (Skeptic's Eye)

Reviewing the top-ranked recommendations manually to verify whether the baseline scoring logic is flagging genuine opportunities or making systematic errors.

In [3]:
top20 = ranked_queue.head(20).copy()

review_cols = [
    'baseline_rank', 'content_id', 'client_id', 'baseline_refresh_score',
    'reason_code', 'action_label', 'impressions_90d', 'avg_position', 'ctr',
    'days_since_last_update', 'is_declining_label'
]

print("=======================================================================================================================")
print("TOP 20 BASELINE RANKED QUEUE PREVIEW")
print("=======================================================================================================================")
print(top20[review_cols].to_string(index=False))

print("\n" + "="*120)
print("TOP-10 SKEPTIC HAND REVIEW (ONE LINE EACH)")
print("="*120)

reviews = [
    (1, "content_9532f197bbc8", "refresh", "page_one_decay_risk", "Rank 2.0 on Page 1 with 309k impressions; 104d un-updated. Declining (1). Right choice — high risk page."),
    (2, "content_4d1fe5b32dc2", "refresh", "page_one_decay_risk", "Rank 2.5 on Page 1 with 98k impressions. Not declining (0). FALSE POSITIVE — Stable top position core page."),
    (3, "content_07f2e7a6f38a", "refresh", "page_one_decay_risk", "Rank 2.7 on Page 1 with 101k impressions; CTR 0.85%. Not declining (0). Wrong if high CTR is optimal for keyword."),
    (4, "content_e5ae436f9a16", "refresh_and_review_ctr", "low_ctr_visible_page", "Rank 3.0 with 117k impressions and 0.45% CTR. Not declining (0). Wrong if title tag already matches SERP intent."),
    (5, "content_3430a8b94511", "refresh_and_review_ctr", "low_ctr_visible_page", "Rank 3.3 with 152k impressions and 0.29% CTR. Declining (1). TRUE POSITIVE — Severe low CTR on massive volume."),
    (6, "content_cbd93118300b", "refresh_and_review_ctr", "low_ctr_visible_page", "Rank 3.3 with 145k impressions and 0.46% CTR. Declining (1). TRUE POSITIVE — High volume page undergoing traffic decay."),
    (7, "content_9c195417f6ef", "refresh", "page_one_decay_risk", "Rank 2.5 with 79k impressions; 104d old. Not declining (0). Wrong if content topic is evergreen."),
    (8, "content_ba2acb4ebd04", "refresh", "page_one_decay_risk", "Rank 3.6 with 142k impressions and 0.83% CTR. Not declining (0). Wrong if impressions are growing naturally."),
    (9, "content_79b25654070a", "refresh_and_review_ctr", "low_ctr_visible_page", "Rank 3.7 with 148k impressions and 0.48% CTR. Not declining (0). Wrong if keyword has high zero-click search intent."),
    (10, "content_adddad39251c", "refresh", "page_one_decay_risk", "Rank 3.6 with 129k impressions. Not declining (0). Wrong if recent Google update already re-indexed the page.")
]

for rank, cid, act, reason, note in reviews:
    print(f"Rank {rank:02d} [{cid}] | Action: {act:22s} | Reason: {reason:20s} | Review: {note}")

TOP 20 BASELINE RANKED QUEUE PREVIEW
 baseline_rank           content_id         client_id  baseline_refresh_score          reason_code           action_label  impressions_90d  avg_position  ctr  days_since_last_update  is_declining_label
             1 content_9532f197bbc8 client_4e07408562                0.941189  page_one_decay_risk                refresh           309192           2.0 0.87                     104                   1
             2 content_4d1fe5b32dc2 client_19581e27de                0.934889  page_one_decay_risk                refresh            97999           2.5 0.52                     104                   0
             3 content_07f2e7a6f38a client_19581e27de                0.934080  page_one_decay_risk                refresh           101078           2.7 0.85                     104                   0
             4 content_e5ae436f9a16 client_4e07408562                0.933606 low_ctr_visible_page refresh_and_review_ctr           117741           3.0 0.

## 4. Weak picks + leakage check

### Analysis of Weak Picks (False Positives in Top 20)

- **Weak Pick Example 1**: `content_4d1fe5b32dc2` (Rank #2). Scores 0.935 with 98k impressions at rank 2.5, but its `is_declining_label` is `0`. The baseline over-weights raw impression volume and position 2.5 as "decay risk" even when traffic is completely steady.
- **Weak Pick Example 2**: `content_e5ae436f9a16` (Rank #4). Scores 0.934 with 117k impressions and 0.45% CTR. However, for certain navigational queries, a 0.45% CTR is baseline normal. The fixed threshold of 0.5% flags pages that are not actually declining.

### Data & Feature Leakage Audit

- **Leakage Check**: Confirmed that `trend_direction` and `trend_pct` are **NEVER** included in feature engineering or score calculation.
- **Time-Window Check**: Features use trailing-90-day aggregate metrics (`impressions_90d`, `days_since_last_update`, `avg_position`, `ctr`, `word_count`). No future-window lookaheads or label-derived variables are present in the score inputs.

In [4]:
# Verify zero correlation between features and label-derived leak columns
leak_columns = ['trend_direction', 'trend_pct']

print("==========================================================================")
print("LEAKAGE AUDIT: Checking score inputs for target/future window contamination")
print("==========================================================================")

score_features = ['impressions_90d', 'days_since_last_update', 'avg_position', 'ctr', 'word_count']
for f in score_features:
    assert f not in leak_columns, f"LEAK ERROR: Feature {f} is derived from target!"
print("PASSED: Zero leakage features detected in baseline score calculation.")

# Display false positive rate in top 20
top20_fp = (top20['is_declining_label'] == 0).sum()
print(f"Top-20 False Positives: {top20_fp} / 20 ({top20_fp/20*100:.1f}%)")
print("Conclusion: Hand-written rules over-predict on high-volume steady pages. ML models in W5 must learn non-linear interactions to reduce these false positives.")

LEAKAGE AUDIT: Checking score inputs for target/future window contamination
PASSED: Zero leakage features detected in baseline score calculation.
Top-20 False Positives: 13 / 20 (65.0%)
Conclusion: Hand-written rules over-predict on high-volume steady pages. ML models in W5 must learn non-linear interactions to reduce these false positives.


## Self-check

Before submitting, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.